# Speech-to-Text (STT) with Whisper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nontgcob/HKU-InnoWing-STT-TTS-Workshop/blob/main/src/stt.ipynb)

This notebook demonstrates Speech-to-Text (STT), also known as Automatic Speech Recognition (ASR), with the Whisper model from Hugging Face Transformers.

The workshop focus is browser microphone transcription in Google Colab. A secondary file-based path is also included so you can reuse the same transcription pipeline with your own audio files.

Note: the browser microphone flow below depends on Google Colab and browser microphone permissions.


In [ ]:
%pip install -q "transformers>=5.0.0" "torch>=2.7.1" "librosa>=0.11.0" "soundfile>=0.13.1" "sentencepiece>=0.2.1"

## 1. Load the Whisper Model

We will use `openai/whisper-small`, which is a good balance between speed and transcription quality for a workshop notebook.


In [ ]:
from pathlib import Path

import base64
import threading

import librosa
import soundfile as sf
from IPython.display import Audio, Javascript, display
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")


def load_audio(path, target_sr=16000):
    audio, sample_rate = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sample_rate != target_sr:
        audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=target_sr)
    return audio


def transcribe_audio_file(path, sampling_rate=16000):
    audio = load_audio(path, target_sr=sampling_rate)
    input_features = processor.feature_extractor(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt",
    ).input_features
    predicted_ids = model.generate(input_features)
    transcription = processor.tokenizer.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0]
    return transcription

## 2. Optional: Transcribe an Audio File

This path is useful if you already have a WAV file in the Colab runtime. It uses the same preprocessing and transcription flow as the microphone recording path.


In [ ]:
audio_file_path = Path("recording.wav")

if audio_file_path.exists():
    file_transcription = transcribe_audio_file(audio_file_path)
    print("--- Transcription Result from Audio File ---")
    print(file_transcription)
    display(Audio(str(audio_file_path)))
else:
    print(
        "Optional step skipped. Upload a file named 'recording.wav' or change "
        "audio_file_path to your own WAV file to use this path."
    )

## 3. Transcribe from Your Browser Microphone in Colab

The next cells create a small JavaScript recorder in the notebook, save the recorded audio to the Colab runtime, and then run the same Whisper transcription pipeline.


In [ ]:
from google.colab import output

recording_done_event = threading.Event()


def record_audio(duration, filename="microphone_audio.wav", samplerate=16000):
    recording_done_event.clear()

    js_code = f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const mediaRecorder = new MediaRecorder(stream);
        const audioChunks = [];

        mediaRecorder.addEventListener('dataavailable', event => {{
            audioChunks.push(event.data);
        }});

        const audioPromise = new Promise(resolve => {{
            mediaRecorder.addEventListener('stop', () => {{
                const audioBlob = new Blob(audioChunks, {{ type: 'audio/wav' }});
                const reader = new FileReader();
                reader.onloadend = () => resolve(reader.result.split(',')[1]);
                reader.readAsDataURL(audioBlob);
                stream.getTracks().forEach(track => track.stop());
            }});
        }});

        mediaRecorder.start();
        await new Promise(resolve => setTimeout(resolve, {duration * 1000}));
        mediaRecorder.stop();

        return await audioPromise;
    }}

    recordAudio().then(audioBase64 => {{
        google.colab.kernel.invokeFunction(
            'notebook_record_audio_callback',
            [audioBase64, '{filename}', {samplerate}],
            {{}}
        );
    }});
    """

    def _record_audio_callback(audio_base64, callback_filename, callback_samplerate):
        audio_bytes = base64.b64decode(audio_base64)
        with open(callback_filename, "wb") as f:
            f.write(audio_bytes)

        print(f"Audio saved to {callback_filename}")
        print("Transcribing recorded microphone audio...")
        transcription = transcribe_audio_file(
            callback_filename, sampling_rate=callback_samplerate
        )

        print("\n--- Transcription Result from Microphone ---")
        print(transcription)
        print("\n--- Playing back recorded audio ---")
        display(Audio(callback_filename))
        recording_done_event.set()

    output.register_callback("notebook_record_audio_callback", _record_audio_callback)
    display(Javascript(js_code))
    print(f"Recording for {duration} seconds...")

    timeout_seconds = duration + 15
    if not recording_done_event.wait(timeout=timeout_seconds):
        print(
            "Warning: recording or transcription timed out. Check Colab browser "
            "permissions and try again."
        )

In [ ]:
recording_duration_seconds = 5
mic_audio_path = "microphone_audio.wav"
record_audio(duration=recording_duration_seconds, filename=mic_audio_path)